# 01 – Running the FinQA experiments

**Seminar paper:** Benchmarks for Financial LLM Agents. This notebook covers the empirical part (step 1 of 2).

This notebook queries the local models through Ollama and saves every raw answer to `results/*.jsonl`.
Scoring and all tables and figures are in **`02_analysis.ipynb`**, which only reads these files and runs in seconds.

**Design**
- **Data:** FinQA test set (1,147 questions, the same split FinBen uses); fixed random sample of 200 questions (seed 42).
- **Models:** `qwen3.5:9b` (dense), `gemma4:26b` and `qwen3-coder:30b` (mixture-of-experts). Thinking mode off.
- **Settings:**
  - `cot`: chain-of-thought, final line `Answer: <number>`
  - `tool`: the model writes Python, the notebook executes it, and on an error the model gets one repair round (a minimal agent loop: act → observe → fix)
- **Decoding:** temperature 0, seed 42, context window 4,096 tokens.

**Before you start**
1. Start Ollama (`ollama serve`, or open the Ollama app).
2. Install the packages: `pip install -r requirements.txt`
3. Close other apps. With 24 GB of RAM, the 26–30B models only just fit.

**Resuming:** every answer is written to disk immediately. If the run is interrupted (sleep, crash, Ctrl-C), simply run the cell again; finished questions are skipped.

⚠️ In the `tool` setting, the notebook runs code written by the model on your machine (in a separate process with a 10-second timeout). That's fine for FinQA arithmetic, but it isn't a security sandbox.

In [ ]:
# ---- Configuration ----
MODELS = ["qwen3.5:9b", "gemma4:26b", "qwen3-coder:30b"]
SETTINGS = ["cot", "tool"]

N_QUESTIONS = 200   # main sample
N_PILOT = 20        # pilot run to measure speed
SEED = 42

CHAT_OPTIONS = dict(num_ctx=4096, num_predict=1024, seed=SEED, think=False)

In [ ]:
import json
import pandas as pd
import requests
import finqa_eval as fe

items = fe.load_finqa("data")
sample = fe.sample_items(items, N_QUESTIONS, seed=SEED)
pilot = sample[:N_PILOT]
print(f"FinQA test set: {len(items)} questions, sample: {len(sample)}, pilot: {len(pilot)}")
print("yes/no questions in sample:", sum(isinstance(it["gold_exe"], str) for it in sample))

# Save the sample IDs, so the sample is documented and reproducible
pd.Series([it["id"] for it in sample], name="id").to_csv("data/sample_ids.csv", index=False)

### Example prompt (tool setting)

In [ ]:
print(fe.build_prompt(sample[0], "tool"))
print("\nGold answer:", sample[0]["gold_answer"], "| exact:", sample[0]["gold_exe"], "| program:", sample[0]["gold_program"])

### Is Ollama running, and are all models installed?

In [ ]:
installed = {m["name"] for m in requests.get(f"{fe.OLLAMA_URL}/api/tags", timeout=10).json()["models"]}
for m in MODELS:
    print(f"{m:20s}", "OK" if m in installed else "MISSING – run: ollama pull " + m)

## Step 1: Pilot (20 questions per model and setting)

Measures the real speed on your laptop. The pilot results go to a separate folder (`results_pilot/`) and are not used in the paper.

While it runs, check `ollama ps` in a terminal: **`100% GPU`** means the model fits completely into GPU memory.

In [ ]:
# One model at a time; each model is unloaded before the next one is loaded
for model in MODELS:
    for setting in SETTINGS:
        fe.run_experiment(pilot, model, setting, results_dir="results_pilot", **CHAT_OPTIONS)
    fe.unload_model(model)

In [ ]:
rows = []
for model in MODELS:
    for setting in SETTINGS:
        path = fe.results_path("results_pilot", model, setting)
        recs = [json.loads(l) for l in path.read_text().splitlines()] if path.exists() else []
        if not recs:
            continue
        by_id = {it["id"]: it for it in pilot}
        sec = sum(r["seconds"] for r in recs) / len(recs)
        rows.append({
            "model": model, "setting": setting, "n": len(recs),
            "s/question": round(sec, 1),
            f"est. hours for {N_QUESTIONS}": round(sec * N_QUESTIONS / 3600, 2),
            "accuracy (tolerant)": sum(fe.score_tolerant(r["final"], by_id[r["id"]]) for r in recs) / len(recs),
            "no final answer": sum(r["final"] is None for r in recs),
            "truncated prompts": sum(r["truncated_prompt"] for r in recs),
        })
pilot_df = pd.DataFrame(rows)
print(f"Estimated total time for the main run: {pilot_df[f'est. hours for {N_QUESTIONS}'].sum():.1f} h")
pilot_df

**Check before the main run**
- *s/question* above ~45 s for a model → it doesn't fit into memory; consider a smaller quantization.
- *no final answer* high → look at the raw responses (cell below); the prompt or parser may need a fix.
- *truncated prompts* > 0 → increase `num_ctx` in the configuration (costs memory).

In [ ]:
# Look at a few raw pilot responses
path = fe.results_path("results_pilot", MODELS[0], "cot")
for r in [json.loads(l) for l in path.read_text().splitlines()][:2]:
    print(r["response"][-600:], "\n→ extracted:", r["final"], "\n" + "-" * 80)

## Step 2: Main run (200 questions)

Runs one model at a time (both settings), so Ollama doesn't keep reloading models. Plug in the laptop and turn off sleep mode.
To spread the run over several nights, set `MODELS_TO_RUN` to a single model.

In [ ]:
MODELS_TO_RUN = MODELS   # e.g. ["qwen3.5:9b"] for one model per night

# One model at a time; each model is unloaded before the next one is loaded
for model in MODELS_TO_RUN:
    for setting in SETTINGS:
        fe.run_experiment(sample, model, setting, results_dir="results", **CHAT_OPTIONS)
    fe.unload_model(model)

### Completeness check

In [ ]:
for model in MODELS:
    for setting in SETTINGS:
        path = fe.results_path("results", model, setting)
        n = len(path.read_text().splitlines()) if path.exists() else 0
        print(f"{model:20s} {setting:5s} {n:4d}/{N_QUESTIONS}", "✓" if n == N_QUESTIONS else "← not finished, rerun step 2")

Once everything is complete, continue with **`02_analysis.ipynb`**.